# Simulating activation blockages

In this notebook we are going to play with activation blocakges. Activation blockages can happen at tissue heterogeneities, due to differences in action potential duration (APD).

We are going to configure some cases that exemplify this behaviour.

Like in the previous examples, the first step is to install the arritmic3d module from the source code repository (installation from PyPi, soon). We will also import the main dependencies we need.



In [ ]:
%pip install arritmic3d

In [ ]:
import os
import time
import shutil
from google.colab import files
import pyvista as pv
import matplotlib.pyplot as plt

# Flag to download the simulated cases.
# Set to True if you want the results to
# be downloaded, for inspection with paraview
download_cases = True

## Plotting functions

Before we proceed, we will define some auxiliary functions that we will later use to visualize the results.

In [ ]:
def plot_grid(grid,field="AP",plt_show=False,title = "") :
    rng = ranges.get(field,None)
    # Apply a threshold to the grid. Cells with restitution_model==0
    # are not part of the simulation domain
    grid = grid.threshold(0.5, scalars="restitution_model", all_scalars=True)
    # Setup plotter for a static image
    plotter = pv.Plotter(off_screen=True)
    plotter.add_mesh(grid, scalars=field, cmap="coolwarm", show_edges=True,rng=rng)
    plotter.view_xy()
    img = plotter.show(screenshot=True)
    # Display using matplotlib
    plt.imshow(img)
    plt.axis('off')
    if title != "":
        plt.title(title)
    if plt_show:
        plt.show()
    return img

def plot_vtk(file_path, field="AP",plt_show=False,title = ""):
    grid = pv.read(file_path)
    plot_grid(grid,field=field,plt_show=plt_show,title=title)

def plot_animation(case_dir, field="AP", init_time=None, end_time=None, step=None):
    config = a3d.load_case_config(case_dir)
    if init_time is None:
        init_time = config.get("VTK_OUTPUT_INITIAL_TIME")
    if step is None:
        step = config.get("VTK_OUTPUT_PERIOD")
    if end_time is None:
        end_time = config.get("SIMULATION_DURATION")

    for time_ms in range(init_time, end_time + 2, step):
        file_path = f"{case_dir}/slab_{time_ms:05d}.vtu"
        if os.path.exists(file_path):
            plot_vtk(file_path, field=field, plt_show=True, title=f"Time: {time_ms} ms")
            # Small pause to allow the UI to update
            time.sleep(0.01)

ranges = {
    "AP": [-80, 40],
    "State": [0, 2]
}

def download_case(case_path):
    print(" Packaging results...")
    if os.path.exists(case_path) and len(os.listdir(case_path)) > 0:
        zip_name = "/content/case.zip"
        !zip -q -FSr {zip_name} {case_path} > /dev/null
        print(" Downloading results...")
        files.download(zip_name)
        print(f" DOWNLOAD STARTED: {zip_name}")
    else:
        print(" Error: Simulation produced no files.")

## Blockage in the frontier with border zone

Our first test case will be a slab with a square of BZ in the middle.

In [ ]:
# We import Arritmic3d
import arritmic3d as a3d

In [ ]:

# We start by defining the output directory
output_dir="test_slab"

# Remove the directory if it exists (for notebook repeatability)
if os.path.exists(output_dir):
    shutil.rmtree(output_dir)

# Ensure the subfolder for the slab exists and set the output file name
os.makedirs(os.path.join(output_dir, "input_data"), exist_ok=True)
slab_path = os.path.join(output_dir, "input_data", "slab.vtk")

# Define the list of arguments.
# Check https://commlabuv.github.io/arritmic3d/#generating-tissue-slabs
slab_args = [
    slab_path,
    "--nnodes", "70", "70", "2",
    "--spacing", "0.1", "0.1", "0.1",
    "--region-by-side", "south", "1",
    "--field", "restitution_model", "2",
    "--region", '{"shape" : "square", "cx" : 3.5, "cy" : 3.5, "r1" : 2.0, "r2" : 2.0, "restitution_model" : 5}'
]

# Now, we can build the slab with these arguments
a3d.build_slab(args_list=slab_args)

# We plot the slab
plot_vtk(slab_path,field="restitution_model")


Next, we define a fixed CL pacing, with 7 stimuli every 500ms (first beat at t=0 and last beat at t=3000). After the simulation, we plot the AP at t=3010ms and at t=3350ms.

You are encouraged to download the result and animate it using Paraview.

S1-S2 protocol with the following properties:

```
Pacing protocol:
  S1: N_STIMS=6, BCL=600
  S2: N_STIMS=3, BCL=400
  ```


In [ ]:
# Configure the simulation
print("\n--- Configuring simulation ---")
config = {
    "VTK_INPUT_FILE": slab_path,
    "APD_MODEL": "TenTuscher",
    "CV_MODEL": "TenTuscher",
    "CV_MEMORY_COEFF": 0.05,
    "SIMULATION_DURATION": 4000,
    "VTK_OUTPUT_PERIOD": 1,
    "VTK_OUTPUT_INITIAL_TIME": 3000,
    "PROTOCOL": [
        {
            "ACTIVATION_REGION": 1,
            "FIRST_ACTIVATION_TIME": 0,
            "N_STIMS_PACING": [7],
            "BCL": [500]
        }
    ]
}

# Run the simulation
a3d.arritmic3d(output_dir,config)

# Download the simulation if requested
if download_cases:
    download_case(output_dir)



In [ ]:
# Plot two snapshots
plot_vtk(os.path.join(output_dir,"slab_03010.vtu"), plt_show = True)
plot_vtk(os.path.join(output_dir,"slab_03350.vtu"))

You can see that the central BZ region remains active after the surroundng tissue has repolarizes. Thus, what if the tissue activates again before the BZ repolarizes?

Lets add an additional beat 350ms after the last one. This corresponds to an S1-S2 protocol with the following properties:

```
Pacing protocol:
  S1: N_STIMS=7, BCL=500
  S2: N_STIMS=1, BCL=350
  ```


In [ ]:
# Reconfigure the simulation
# We only need to change the "PROTOCOL" field
print("\n--- Reconfiguring simulation ---")
config["PROTOCOL"] = [
        {
            "ACTIVATION_REGION": 1,
            "FIRST_ACTIVATION_TIME": 0,
            "N_STIMS_PACING": [7,1],
            "BCL": [500,350]
        }
    ]

# Run the simulation
a3d.arritmic3d(output_dir,config)

# Download the simulation if requested
if download_cases:
    download_case(output_dir)



If we plot some states, we see that the activation is too early to propagate into the BZ region.

In [ ]:
plot_animation(output_dir,
               field="AP",
               init_time=3355,
               end_time=3385,
               step=10)

## Backwards depolarization of BZ

In this last simulation, the activation was blocked in the complete frontier of the central BZ square. However, if the last stimulus had happened a little bit later, perhaps it would have reached the upper part of the BZ square already depolarized. Let's try changing S2.

In the next code, run the simulation for S2=370. Then, change the value of S2 in the range between 370 and 380 to see what happens. It is recommended to download the simulation data and explore the simulation with Paraview.

In [ ]:
# Reconfigure the simulation
# We only need to change the "PROTOCOL" field
print("\n--- Reconfiguring simulation ---")

S2 = 370 # <- Try values between 370 and 380

config["PROTOCOL"] = [
        {
            "ACTIVATION_REGION": 1,
            "FIRST_ACTIVATION_TIME": 0,
            "N_STIMS_PACING": [7,1],
            "BCL": [500, S2 ]
        }
    ]

# Run the simulation
a3d.arritmic3d(output_dir,config)

plot_animation(output_dir,
               field="AP",
               init_time=3380,
               end_time=3600,
               step=40)

# Download the simulation if requested
if download_cases:
    download_case(output_dir)



# Causing a reentry

In the last simulations, we managed to force a backwards activation of the BZ, after a blockage, for S2 values in the range 375-380. The central square activated completely but the surrounding healthy tissue was depolarized. Thus, after the slab activated completely, it repolarized again after a few ms.

But, what if the depolarization of BZ took long enough to reach part of the healthy tissue already repolarized? In that case, a reentry would take place; the activation _reenters_ the previously activated tissue that is ready for depolariation again.

If you played enough with the values of S2, you probably noticed that the BZ is to small (or the conduction velocity to high) to lead to a reentry. No matter what S2 we use, if it allows BZ activation, the central square activates completely before the surrounding tissue is ready to actiavte again.

However, if the combination of the size of BZ and CV is adequate, a reentry can happen.

Next, we set a completely new case, with a larger slab and a slightly reduced CV.

We start by building the slab. It has the same structure as the previous example, but it is larger.

In [ ]:
# We start by defining the output directory
output_dir="reentry"

# Remove the directory if it exists (for notebook repeatability)
if os.path.exists(output_dir):
    shutil.rmtree(output_dir)

# Ensure the subfolder for the slab exists and set the output file name
os.makedirs(os.path.join(output_dir, "input_data"), exist_ok=True)
slab_path = os.path.join(output_dir, "input_data", "slab.vtk")

slab_path = os.path.join(output_dir, "input_data", "slab.vtk")
slab_args = [
    slab_path,
    "--nnodes", "60", "60", "2",
    "--spacing", "0.3", "0.3", "0.3",
    "--region-by-side", "south", "1",
    "--field", "restitution_model", "2",
    "--region", '{"shape" : "square", "cx" : 9.0, "cy" : 9.0, "r1" : 6.0, "r2" : 6.0, "restitution_model" : 5}'
]

# Build and save the slab
a3d.build_slab(args_list=slab_args, save=True)
print("Slab generated successfully!")

# Visualize the slab
plot_vtk(slab_path, field="restitution_model", plt_show=True, title="Slab")


Then, we define the model properties. Take a look at the parameters and check their meaning in the documentation at [github](https://commlabuv.github.io/arritmic3d/).

Note that, in the stimulation protocol, we need to S2 activations. Check why in the output using paraview.

Run the simulation and, go to the next cell to visualize the activation after the second S2 stimulus.

In [ ]:
# --- Configure the simulation ---
print("\n--- Configuring simulation ---")
config = {
    "VTK_INPUT_FILE": slab_path,
    "APD_MODEL": "TenTuscher",
    "CV_MODEL": "TenTuscher",
    "ELECTROTONIC_EFFECT": 0.0,
    "CV_MEMORY_COEFF": 0.05,
    "CORRECTION_FACTOR_CV": 0.9,
    "APD_MEMORY_COEFF": 0.0,
    "SIMULATION_DURATION": 4800,
    "VTK_OUTPUT_PERIOD": 1,
    "VTK_OUTPUT_INITIAL_TIME": 3000,
    "PROTOCOL": [
        {
            "ACTIVATION_REGION": 1,
            "FIRST_ACTIVATION_TIME": 0,
            "N_STIMS_PACING": [8,2],
            "BCL": [420,320] # 319, block before reentry. 322 late for reentry
        }
    ]
}


# --- STEP 4: Run the simulation ---
print("\n--- Running simulation ---")
a3d.arritmic3d(output_dir, config=config)
print("Simulation completed!")


Now, visualize the result.

In [ ]:
plot_animation(output_dir,
               field="AP",
               init_time=3800,
               end_time=4200,
               step=50)

Often, the range of values at which reentries can happen (often known as vulnerability windows) are quite narrow. Try values around the current value of S2 = 320 to identify which interval of S2 leads to a reentry.